<a href="https://colab.research.google.com/github/Ena-AlexBrush/Fine-Tuning-Experiments/blob/main/GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install datasets evaluate transformers[sentencepiece]

In [24]:
!pip install --upgrade torchao

In [25]:
!pip install trl[GRPOTrainer]

In [26]:
!pip install trl[vllm]

  Using cached vllm-0.25.1-cp38-abi3-manylinux_2_28_x86_64.whl.metadata (11 kB)
Using cached vllm-0.25.1-cp38-abi3-manylinux_2_28_x86_64.whl (250.1 MB)


In [27]:
# # Uninstall current vLLM and potentially conflicting CUDA runtime packages
# !pip uninstall -y vllm nvidia-cuda-runtime-cu12 nvidia-cuda-runtime-cu13
# !pip uninstall -y trl


In [28]:
# Reinstall TRL (without vllm extra to avoid dependency conflicts)
!pip install trl[GRPOTrainer]

# # Install vLLM specifically for CUDA 12.1 (adjust version/CUDA target if needed)
# # For example, vLLM 0.2.7 is known to have a cu121 wheel.
# # Check https://docs.vllm.ai/en/latest/getting_started/installation.html for latest compatible wheels.
# !pip install vllm==0.2.7 --index-url https://download.vllm.ai/whl/cu121

In [29]:
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
import re

In [30]:
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from trl.rewards import accuracy_reward
import re

In [31]:
# train_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="train[:200]") # Using the train dataset.
# eval_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="test[:200]") # Using the test split for evaluation
# model_name = "HuggingFaceTB/SmolLM2-135M" # The model

# # Function to extract prompt and completion from 'messages'
# def extract_prompt_completion_from_messages(item):
#     prompt = None
#     completion = None
#     for message in item['messages']:
#         if message['role'] == 'user':
#             prompt = message['content']
#         elif message['role'] == 'assistant':
#             completion = message['content']
#     return {'prompt': prompt, 'completion': completion}

# # Create a mapping from prompt to gold completion for ground truth checking
# prompt_to_gold_completion_map = {}
# for item in train_dataset:
#     extracted = extract_prompt_completion_from_messages(item)
#     if extracted['prompt'] is not None and extracted['completion'] is not None:
#         prompt_to_gold_completion_map[extracted['prompt']] = extracted['completion']

# # Apply the extraction to create 'prompt' and 'completion' columns in the datasets
# train_dataset = train_dataset.map(extract_prompt_completion_from_messages, remove_columns=['messages'])
# eval_dataset = eval_dataset.map(extract_prompt_completion_from_messages, remove_columns=['messages'])

# # Filter out examples where prompt or completion is None after extraction
# train_dataset = train_dataset.filter(lambda x: x['prompt'] is not None and x['completion'] is not None)
# eval_dataset = eval_dataset.filter(lambda x: x['prompt'] is not None and x['completion'] is not None)

In [32]:
print(train_dataset[0].keys())


dict_keys(['category', 'difficulty', 'quality', 'reward_model_score', 'conversation_tokens', 'prompt', 'completion'])


In [33]:
# # Reward based on matching ground truth
# def ground_truth_check_reward(prompts, completions, **kwargs):
#     rewards = []
#     reward_fn_kwargs = kwargs.get('reward_fn_kwargs', {})
#     prompt_to_gold_completion_map = reward_fn_kwargs.get('prompt_to_gold_completion_map', {})

#     for prompt, completion in zip(prompts, completions):
#         gold_completion = prompt_to_gold_completion_map.get(prompt, None)
#         if gold_completion and completion == gold_completion:
#             rewards.append(1.0)
#         else:
#             rewards.append(0.0)
#     print(f"\n--- ground_truth_check_reward Debug ---")
#     print(f"Prompts: {prompts[:5]}...") # Print first 5 prompts
#     print(f"Completions: {completions[:5]}...") # Print first 5 completions
#     print(f"Truth Rewards: {rewards[:5]}...") # Print first 5 rewards
#     print(f"---------------------------------------")
#     return rewards

# def combined_reward_function(prompts, completions, **kwargs):
#     format_rewards = reward_format(completions, **kwargs)
#     truth_rewards = ground_truth_check_reward(prompts, completions, **kwargs)

#     combined = [ (f_r + t_r) / 2.0 for f_r, t_r in zip(format_rewards, truth_rewards) ]
#     print(f"\n--- combined_reward_function Debug ---")
#     print(f"Combined Rewards: {combined[:5]}...") # Print first 5 combined rewards
#     print(f"--------------------------------------")
#     return combined

In [43]:
import re
from datasets import load_dataset

# 1. Dataset Preparation
train_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="train[:200]")
eval_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="test[:200]")
model_name = "HuggingFaceTB/SmolLM2-135M"

def extract_prompt_completion_from_messages(item):
    prompt = None
    completion = None
    for message in item['messages']:
        if message['role'] == 'user':
            prompt = message['content']
        elif message['role'] == 'assistant':
            completion = message['content']
    return {'prompt': prompt, 'completion': completion}

prompt_to_gold_completion_map = {}
for item in train_dataset:
    extracted = extract_prompt_completion_from_messages(item)
    if extracted['prompt'] is not None and extracted['completion'] is not None:
        prompt_to_gold_completion_map[extracted['prompt']] = extracted['completion']

train_dataset = train_dataset.map(extract_prompt_completion_from_messages, remove_columns=['messages'])
eval_dataset = eval_dataset.map(extract_prompt_completion_from_messages, remove_columns=['messages'])

train_dataset = train_dataset.filter(lambda x: x['prompt'] is not None and x['completion'] is not None)
eval_dataset = eval_dataset.filter(lambda x: x['prompt'] is not None and x['completion'] is not None)


# 2. INSERTED: Format Reward Function
def reward_format(completions, **kwargs):
    """
    Rewards completions that contain a proper XML structure,
    such as <think>...</think> and <answer>...</answer> tags.
    """
    rewards = []
    # Pattern to check for matching open and close tags
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"

    for completion in completions:
        # Clean up whitespace and check against the pattern
        clean_completion = completion.strip()
        if re.match(pattern, clean_completion, re.DOTALL):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


# 3. Ground Truth Reward Function
def ground_truth_check_reward(prompts, completions, **kwargs):
    rewards = []
    reward_fn_kwargs = kwargs.get('reward_fn_kwargs', {})
    prompt_to_gold_completion_map = reward_fn_kwargs.get('prompt_to_gold_completion_map', {})

    for prompt, completion in zip(prompts, completions):
        gold_completion = prompt_to_gold_completion_map.get(prompt, None)
        if gold_completion and completion == gold_completion:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


# 4. Combined Reward Function
def combined_reward_function(prompts, completions, **kwargs):
    format_rewards = reward_format(completions, **kwargs)
    truth_rewards = ground_truth_check_reward(prompts, completions, **kwargs)

    combined = [ (f_r + t_r) / 2.0 for f_r, t_r in zip(format_rewards, truth_rewards) ]
    return combined


In [44]:
# training config
training_args = GRPOConfig(
    output_dir="output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    logging_steps=10,
)

In [45]:
# more detailed config (from the website)
training_args = GRPOConfig(
    output_dir="output",
    num_train_epochs=3,
    num_generations=4,  # Number of completions to generate for each prompt
    per_device_train_batch_size=4,  # We want to get all generations in one device batch
    # Optional but useful
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    logging_steps=10,
    # GRPO specific (optional)
    use_vllm=False  # Speed up generation, need to 'pip install trl[vllm]'. Disabled due to CUDA library error.
)

In [46]:
# train_grpo.py
trainer = GRPOTrainer(
    model=model_name,
    args=training_args,
    reward_funcs=combined_reward_function,
    train_dataset=train_dataset,
    # reward_kwargs={'reward_args': {'prompt_to_gold_completion_map': prompt_to_gold_completion_map}}
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000
